Come sappiamo in IR l'obiettivo ultimo è riuscire a restituire **documenti rilevanti**, ossia che soddisfino il bisogno informativo dell'utente.

Abbiamo anche visto che per valutare le performance di un sistema di IR è necessario avere a disposizione un **benchmark** composto da tre elementi fondamentali:
1. **Collezione di documenti**
2. **Un insieme di query di test**
3. **Un giudizio di rilevanza per ogni coppia query-documento**

Abbiamo studiato diverse misure di valutazione, le tre base sono:
- **Precision**: misura quanti documenti tra quelli recuperati sono rilevanti:
    $$Precision = \frac{\text{documenti rilevanti}}{\text{documenti recuperati}} = \frac{TP}{TP + FP}$$
- **Recall**: misura quanti documenti rilevanti tra tutti i documenti della collezione sono stati recuperati:
    $$Recall = \frac{\text{documenti rilevanti}}{\text{documenti rilevanti totali}} = \frac{TP}{TP + FN}$$
- **F1-score**: è la media armonica tra precision e recall. Rappresenta una specie di soft minimum, in quanto la media armonica tende ad essere bassa anche se uno solo dei due valori è basso:
    $$F1 = \frac{2PR}{P + R}$$

**Problema**: spesso precision e recall sono in conflitto tra loro, se voglio aumentare molto la recall recupero più documenti, ma rischio così di includere molti documenti non rilevanti abbassando la precision e viceversa.

In questa lezione ci concentreremo in particolare su **come migliorare la Recall**. In questo senso il problema che si affronta è quello per cui un utente potrebbe scrivere una query troppo semplice, usando parole diverse da quelle presenti nei documenti rilevanti e quindi non riuscire a recuperarli con un sistema di retrieval semplice.

es. query: "aircraft" --> un documento rilevante potrebbe contenere "plane", ma non "aircraft". Dal momento che i sistemi di IR visti finora si basano strettamente sul match dei termini, allora questo non riuscirebbe a recuperare quel documento nonostante sia effettivamente rilevante per il bisogno informativo dell'utente.

**Come recuperare documenti rilevanti anche quando non contengono esattamente le parole della query originale?** La strategia che vedremo per risolvere il problema è la **query expansion**: l'idea è **espandere la query iniziale aggiungendo nuovi termini utili**

es. di prima: se l'utente cerca aircraft, il sistema potrebbe aggiungere automaticamente alla query plane, airplane, aviation, flight etc..

Esistono due modi principali per fare questa espansione:
1. **Metodo Locale == Relevance Feedback**: il sistema guarda quali documenti l'utente ha giudicato rilevanti e li sfrutta per migliorare la query
2. **Metodo Globale == Global Query Expansion**: il sistema sfrutta l'intera collezione di documenti in una volta sola, per costruire un **thesaurus** a partire dal quale espandere le query.

**Thesaurus** == una rete di termini che sono in qualche modo correlati tra loro, ad esempio perché sono sinonimi, iperonimi (termine che racchiude in sé termini più specifici, detti iponimi; es. di iperonimo è "animale" che racchiude in sé iponimi come "cane", "gatto", "uccello" etc.) etc..


### **Relevance Feedback**
Il relevance feedback funziona come segue:
1. L'utente fa una query iniziale, spesso breve
2. Il motore di ricerca restituisce una lista di documenti ordinati per rilevanza
3. L'utente marca alcuni documenti come rilevanti e altri come non rilevanti
4. Il sistema usa queste informazioni per espandere la query iniziale, ad esempio aggiungendo termini **importanti** (quelli con alto peso tf-idf per quel documento) che compaiono nei documenti rilevanti
5. Il sistema rilancia la query modificata
6. I nuovi risultati dovrebbero avere recall migliore

Il processo potrebbe anche **essere iterato più volte**: l'utente dà feedback, il sistema espande la query, l'utente dà nuovo feedback etc. fino a quando non si raggiunge un risultato soddisfacente.

es. query iniziale "new space satellite applications". Il sistema restituisce una lista di risultati, da cui l'utente marca alcuni documenti come rilevanti (simbolo +). Dopodiché il sistema guarda il contenuto dei documenti in questione e vede che compaiono termini importanti come "nasa", "launch", "earth" etc.. quindi li utilizza per costruire una query espansa che permetterò di far salire altri documenti rilevanti

<p align="center">
    <img src="img/c11.png" alt="Relevance Feedback" width=45%>
    <img src="img/c12.png" alt="Relevance Feedback" width=40%>
</p>

Prima di passare a capire la teoria dietro il relevance feedback e un algoritmo per implementarlo, è importante sottolineare che questa tecnica come idea generale può funzionare con tutti i modelli che abbiamo visto finora. 

L'algoritmo che vedremo però (Rocchio) è pensato, nella sua forma classica, solo per VSM.

#### Analisi teorica del relevance feedback
Per capire formalmente il relevance feedback, dobbiamo introdurre il concetto di **centroide**. Nel modello vettoriale, sappiamo che ogni documento **è rappresentato come un vettore in uno spazio ad alta dimensionalità, e che ogni dimensione corrisponde a un termine**. 

Quindi ogni documento e la query sono vettori nello spazio, e possiamo confrontarli tra loro usando misure di similarità come la cosine similarity. 

Il **centroide** di un certo insieme di documenti rappresenta **il punto medio di quei documenti nello spazio vettoriale**. Formalmente, sia $D$ un insieme di documenti, allora il centroide di $D$ è dato da:
$$ \mu(D) = \frac{1}{|D|} \sum_{d \in D} \vec{d} $$
Immaginiamo di conoscere tutti i documenti rilevanti ($C_r$) e non rilevanti ($C_{nr}$) per una certa query. Allora la query ideale dovrebbe essere posizionata in modo da essere vicina ai documenti rilevanti e lontana da quelli non rilevanti, quindi intuitivamente dovrebbe essere posizionata in una zona dello spazio che è vicina al centroide dei documenti rilevanti e lontana da quello dei documenti non rilevanti:
$$\vec{q}_\text{opt} = \mu(C_r) - \mu(C_{nr}) $$
Naturalmente però se conoscessimo tutti i documenti rilevanti e non rilevanti sarebbe un cazzo e tutt'uno, avremmo già risolto il problema. Per questo introduciamo l'algoritmo di Rocchio.

#### Algoritmo di Rocchio e Assunzioni di RF
L'algoritmo di Rocchio è il metodo classico per fare relevance feedback in VSM. Si parte dalla query originale $q_0$ e la espande usando **sia i documenti giudicati rilevanti che quelli giudicati non rilevanti**. La formula è la seguente:
$$\vec{q}_m = \alpha \vec{q}_0 + \beta \mu(D_r) - \gamma \mu(D_{nr}) = \alpha \vec{q}_0 + \beta \frac{1}{|D_r|} \sum_{d \in D_r} \vec{d} - \gamma \frac{1}{|D_{nr}|} \sum_{d \in D_{nr}} \vec{d}$$
dove:
- $\vec{q}_m$ è la query modificata
- $\vec{q}_0$ è la query originale
- $D_r$ è l'insieme dei documenti marcati rilevanti
- $D_{nr}$ è l'insieme dei documenti marcati non rilevanti
- $\alpha$, $\beta$, $\gamma$ sono parametri che controllano l'importanza relativa della query originale, dei documenti rilevanti e di quelli non rilevanti

**Quando funziona relevance feedback?** Si enunciano due assunzioni importanti:
1. **Assunzione A1**: *l'utente deve conoscere abbastanza bene il vocabolario della collezione per scrivere una query iniziale decente*.  
Se l'utente infatti usasse parole totalmente diverse da quelle presenti nei documenti, allora già all'inizio il sistema non riuscirebbe a recuperare documenti utili e quindi non avrebbe materiale su cui basarsi per espandere la query.  
es. cosmonaut/astronaut --> se l'utente cerca cosmonaut, ma la collezione usa principalmente astronaut --> problematico.
1. **Assunzione A2**: *i documenti rilevanti devono essere abbastanza simili tra loro*.  
Il relevance feedback infatti funziona bene se i documenti rilevanti condividono termini o argomenti simili, ma può dare problemi se una stessa query ha più "prototipi" diversi di documenti rilevanti.  
es. query = "personaggio con doppia vita". Questa query può avere documenti rilevanti molto diversi tra loro; alcuni potrebbero parlare di Dr. Jekyll e Mr. Hyde, altri di Batman, altri ancora di una spia o un agente segreto. Il problema è che tutti questi documenti non usano necessariamente gli stessi termini --> se l'utente marca come rilevante quello di Batman, il sistema gli suggerirà roba legata a Gotham, Joker, pipistrelli etc.. peggiorando rispetto agli altri argomenti.

#### Problemi del relevance feedback e Pseudo-Relevance Feedback
Il relevance feedback ha molti problemi pratici: 
1. **è costoso**: le query modificate diventano più lunghe, quindi più costose da processare
2. **Gli utenti non sono sempre disposti a dare feedback**: molte persone spesso non hanno voglia di segnare manualmente i risultati rilevanti e non rilevanti rispetto alla loro query
3. **Può essere difficile capire perché un certo documento viene recuperato dopo il feedback**: infatti la query modificata diventa spesso lunga e poco interpretabile

Riguardo il secondo problema, per evitare di chiedere il feedback manuale all'utente, si può usare il **Pseudo-Relevance Feedback**. L'idea è la seguente:
1. l'utente scrive una query 
2. il sistema recupera i primi risultati
3. il sistema **assume automaticamente che i primi $k$ documenti siano rilevanti**, usando quei documenti come se fossero feedback positivo
4. il sistema espande la query e rilancia la ricerca

Il rischio dello Pseudo-Relevance Feedback è che se i primi $k$ documenti recuperati non sono effettivamente rilevanti, allora si rischia di espandere la query in modo errato, peggiorando le performance invece di migliorarle. Si parla in questo senso di **Query Drift**.

Il **Query Drift** avviene quando la query modificata si allontana dal vero bisogno informativo dell'utente. Ad esempio, se l'utente cerca "satellite applications" ma i primi risultati parlano molto di NASA o di clima, allora il sistema può espandere la query con termini che la spostano verso un altro tema. **Ciò può portare la query, dopo una o più iterazioni, a derivare sempre più lontano dal bisogno informativo originale**.

Per questi motivi lo pseudo-relevance feedback può funzionare molto bene in media, ma può anche fare malissimo per alcune query.

### **Query expansion globale**
Come già anticipato, a differenza del Relevance Feedback, la **Global Query Expansion** sfrutta una risorsa costruita a partire dalla collezione del tutto **indipendente dalla singola query dell'utente**.  
Infatti si parla di query expansion globale proprio perché viene fatta una sola volta per tutte, non ogni volta che arriva una query.

Di queste risorse query-independent globali ne esistono tre principali tipi:
1. un **manual thesaurus** costruito a partire dalla collezione, che contiene relazioni di sinonimia, iperonimia etc. tra i termini
2. gli **automatically derived thesaurus**
3. la query equivalence basata sui **query log**

#### Manual Thesaurus
Un **Manual Thesaurus** è una risorsa costruita a "mano" a partire dalla collezione tramite esperti. Essa contiene relazioni tra termini, per esempio (dato un termine) sinonimie, quasi-sinonimie, termini più generici (iperonimi) o più specifici (iponimi), termini correlati etc.

Un contesto in cui questa risorsa è fondamentale è qualsiasi dominio specialistico in cui la terminologia è molto precisa, es. PubMed. In questo caso infatti un utente potrebbe cercare una parola comune come "heart attack", ma i documenti scientifici tipicamente fanno riferimento a questo concetto usando termini più tecnici come "myocardial infarction". Un thesaurus costruito a mano da esperti del dominio può aiutare a collegare questi termini e migliorare la recall.

**Il vantaggio principale del Manual Thesaurus è la qualità**: essendo infatti costruito da esperti del dominio, evita molte associazioni sbagliate che invece potrebbero essere presenti in un thesaurus costruito automaticamente.  
Tuttavia ciò comporta che è anche **molto costoso da creare e mantenere**

**Come funziona in pratica?** Quando l'utente scrive una query, il thesaurus (da immaginare come una sorta di dizionario, dove ad ogni termine è associata una lista di termini correlati) viene consultato per ogni termine della query, e i termini correlati vengono aggiunti alla query originale con un peso minore rispetto a quello dei termini originali.

es. query == "aircraft safety" --> thesaurus "aircraft" --> "plane", "aviation", "flight" etc. --> query espansa == aircraft OR safety OR plane OR aviation OR flight etc. (con pesi minori per i termini aggiunti)

**Questo metodo (così come in generale Global query expansion) aumenta chiaramente la recall** perché permette di recuperare documenti che non contengono esattamente i termini della query originale, ma che sono comunque rilevanti. **Tuttavia può anche abbassare la precision** se vengono aggiunti termini sbagliati o troppo generali che portano a recuperare documenti non rilevanti (rischio di **query drift**).

es. jaguar --> termine polisemico che può riferirsi sia ad un animale che a una marca di auto --> se espando jaguar anche con la marca d'auto ma l'utente voleva l'animale --> rischio di recuperare documenti non rilevanti abbassando la precision

#### Automatically Derived Thesaurus
Invece di costruire manualmente il tesauro, l'idea è **generarlo automaticamente a partire dalla collezione, osservando la distribuzione dei termini nei documenti**.

Infatti banalmente parole simili tendono ad apparire in contesti simili. Cerchiamo anzitutto di capire quando due termini sono simili:
1. **Definizione 1: due parole sono simili se co-occorrono con parole simili**.  
Ad esempio $\text{car} \approx \text{motorcycle}$ perché entrambi co-occorrono con parole come "drive", "engine", "road" etc.. 
2. **Definizione 2: due parole sono simili se entrano nelle stesse relazioni grammaticali**.  
Questa definizione è più raffinata della prima: es. $\text{apples} \approx \text{oranges}$ perché molte frasi possono essere costruite intercabiandole senza perdere di significato (es. "eat apples" --> "eat oranges", "peel apples" --> "peel oranges" etc.) --> **non sto più semplicemente contando parole vicine nella stessa finestra, ma sto guardando proprio la relazione grammaticale tra i termini**. Questa seconda definizione è quindi più linguistica e può essere più precisa, ma è anche più complessa da implementare.

#### Query Logs
La fonte principale di query expansion nei motori di ricerca è rappresentata dai **query logs**. 

Un query log è un *insieme di dati che contiene tutte le query che gli utenti hanno fatto a un motore di ricerca, insieme ai documenti che hanno cliccato e eventualmente alle riforimulazioni della query, fino ai documenti che hanno soddisfatto il loro bisogno informativo.*

es. 1, **query successive**: dopo aver cercato "herbs", tipicamente gli utenti tendono a cercare "herbal remedies" --> "herbal remedies" è una possibile espansione di "herbs" che si può ricavare dai query log.

es. 2, **query diverse che portano allo stesso click**: utenti che cercano "batman" e utenti che cercano "bruce wayne" tendono a cliccare sugli stessi documenti --> "bruce wayne" è una possibile espansione di "batman" e viceversa.

**Differenza importante tra tesauri e query logs**: *mentre i tesauri sono costruiti a partire dai documenti e si basano sulla distribuzione/semantica dei termini nei documenti*, i **query logs si basano invece sul comportamento degli utenti e sulle loro interazioni con il motore di ricerca.**

**In generale** ricorda sempre che la Query Expansion Globale non guarda ai documenti restituiti dalla query corrente, ma si basa su risorse costruite a priori (manuali o automatiche) per espandere la query. 

Chiaramente **non esiste una soluzione universamente migliore**: dipende dalla mia collezione e applicazione. Se ho una collezione molto specialistica, allora un manual thesaurus costruito da esperti del dominio può essere la scelta migliore. Se invece ho una collezione più generale, allora i query logs possono essere più efficaci perché riflettono il comportamento reale degli utenti. Gli automatically derived thesauri possono essere utili in entrambi i casi, ma è importante valutarne la qualità e l'efficacia per la specifica collezione e dominio di interesse.

#### Potenziale domanda aperta esame
**Domanda aperta esame:** arriva una query, voglio applicare query expansion. Quando la applico, **quali termini devo aggiungere, con che peso, e come evito che i termini aggiunti possano ammazzare la precision?**

In questo senso la questione dei **pesi** è centrale affinché la query espansa non diventi una query troppo distante da quella originale (usability). 

Supponiamo che l'utente cerchi $q = \text{"car"}$. Supponiamo che il tesauro/query log o LSI (per LSI e la sua applicazione nella query expansion vedi md successivo) mi suggerisca parole del tipo "automobile", "vehicle", "engine", "road".  
Allora potrei espandere la query come:
$$q' = \text{car automobile vehicle engine road}$$
Tuttavia assegnare lo stesso peso anche ai termini aggiunti è **pericoloso**: la query potrebbe essere spostata nello spazio vettoriale verso termini diversi/troppo generali (es. vehicle e road) e quindi recuperare documenti non rilevanti per l'utente. 

Per questi motivi la query dovrebbe essere espansa tenendo conto di un peso per i nuovi termini, possiamo pensarla con una formula intuitivamente così:
$$q' = \alpha q + \beta e$$
dove $q$ è la query originale, $e$ è il vettore dei termini aggiunti, $\alpha$ è un peso che controlla l'importanza della query originale, e $\beta$ è un peso che controlla l'importanza dei termini aggiunti. Chiaramente $\beta < \alpha$ per evitare di spostare troppo la query rispetto alla sua forma originale.

Quindi per l'esempio, si potrebbe mantenere $\alpha$ a 1 e assegnare a $\beta = 0.5$, si otterrebbe la nuova query come il vettore risultante dalla seguente combinazione lineare:
$$q' = 1 \cdot \text{car} + 0.5 \cdot (\text{automobile} + \text{vehicle} + \text{engine} + \text{road})$$

**Come decidere i pesi?** Dipende dalla fonte dell'espansione: se ad esempio faccio riferimento ad un thesaurus manuale, dove si distinguono i termini anche in base al fatto che siano sinonimi stretti, quasi-sinonimi, termini più generali o correlati rispetto a un termine --> potrei dare pesi più alti per i sinonimi stretti e più bassi rispetto alle altre categorie.

Nel caso dei tesauri automatici, si potrebbe invece usare come peso proprio la similarità tra i termini: es. "automobile" avrà una similarità magari di 0.9 con "car" dal momento che co-occorrono in molti contesti simili, mentre "road" avrà una similarità più bassa, ad esempio 0.3, perché co-occorre con "car" in contesti più generali e meno specifici --> peso adatto in base alla similarità con il nuovo termine.

Nel caso di query log la questione è più complessa, si può pesare l'espansione in base alla frequenza con cui gli utenti si comportano in un certo modo (es. se molti utenti che cercano "car" poi cercano "automobile", allora "automobile" avrà un peso più alto rispetto a "road" che magari viene cercato meno frequentemente dopo "car").

Si ricorda infatti che il rischio che comporta l'abbassamento della Precision sia nel relevance feedback che nella query expansion globale è quello di **query drift**: dopo l'espansione, non si vuole assolutamente che la query si allontani troppo dall'intento originale dell'utente.

Quindi workflow completo:
1. L'utente scrive una query $q$, lo trasformo in un vettore pesato (es. tf-idf). Importante che sia un vettore perché come visto prima, per fare query expansion è necessario assegnare dei pesi ai termini aggiunti, e quindi è necessario che la query sia rappresentata come un vettore in uno spazio ad alta dimensionalità.
2. per ogni termine della query, cerco possibili espansioni in una risorsa globale (manual thesaurus, automatic thesaurus, query log, LSI etc.)
3. in base ai ragionamenti visti sopra, assegno un peso a ciascun termine aggiunto, e costruisco la query espansa $q'$
4. eseguo il retrieval con la query espansa $q'$, ottenendo una lista di documenti ordinati per rilevanza rispetto a $q'$

**N.B** a volte, se il modello che utilizziamo è molto costoso (vedremo ad es. LSI) --> si potrebbe usare query expansion per fare **candidate generation** (si inseriscono nella query i nuovi termini per avere a disposizione più documenti a partire dall'inverted index, che non siano solo documenti che contengono esattamente i termini della query originale), e poi si applica il modello in questione solo su questo sottinsieme di documenti (si parla in questo senso di **re-ranking**).  
In questo modo si evita di dover confrontare la query con tutti i documenti della collezione (operazione molto costosa se modello sofisticato), filtrando comunque documenti candidati che con buonissima probabilità sono rilevanti grazie alla query expansion.